# Loading Data

In [145]:
import pandas as pd
import numpy as np

In [146]:
# The following code will only execute
# successfully when compression is complete

import kagglehub

# Download latest version
path = kagglehub.dataset_download("s1nghayush/ig-filter-data")

print("Path to dataset files:", path)

Path to dataset files: /root/.cache/kagglehub/datasets/s1nghayush/ig-filter-data/versions/2


In [147]:
df = pd.read_csv(path + "/ganjoo_ipl_bbb.csv")

# Cleaning Data

In [148]:
df = df.sort_values(by=['date', 'inns', 'ball_id'])

In [149]:
df.drop(columns=['wagonX', 'wagonY', 'wagonZone', 'line','length', 'shot', 'control', 'predscore', 'wprob'], inplace=True)

In [150]:
print(df.columns)

Index(['p_match', 'inns', 'bat', 'p_bat', 'team_bat', 'bowl', 'p_bowl',
       'team_bowl', 'ball', 'ball_id', 'outcome', 'score', 'out', 'dismissal',
       'p_out', 'over', 'noball', 'wide', 'byes', 'legbyes', 'cur_bat_runs',
       'cur_bat_bf', 'cur_bowl_ovr', 'cur_bowl_wkts', 'cur_bowl_runs',
       'inns_runs', 'inns_wkts', 'inns_balls', 'inns_runs_rem',
       'inns_balls_rem', 'inns_rr', 'inns_rrr', 'target', 'max_balls', 'date',
       'year', 'ground', 'country', 'winner', 'toss', 'competition',
       'bat_hand', 'bowl_style', 'bowl_kind', 'batruns', 'ballfaced',
       'bowlruns', 'bat_out'],
      dtype='object')


# Tranforming Data to bowling scorecard

In [151]:
def calculate_bowling_card(df):
    # Filter out only legal deliveries (exclude wides and no-balls for overs calculation)
    df['legal_delivery'] = (df['noball'] == 0) & (df['wide'] == 0)

    # Ensure wickets do not include run outs
    df['valid_wicket'] = df['out'] & (df['dismissal'] != 'run out')

    # Group by match and bowler
    bowling_card = df.groupby(['p_match', 'p_bowl', 'bowl', 'team_bowl', 'team_bat', 'inns'])

    # Compute required statistics
    summary = bowling_card.agg(
        overs=('legal_delivery', lambda x: f"{x.sum() // 6}.{x.sum() % 6}"),  # Overs in x.y format
        balls=('ball', 'count'),  # Total deliveries (including wides and no-balls)
        wickets=('valid_wicket', lambda x: x.sum()),  # Count of valid wickets (excluding run outs)
        runs=('bowlruns', 'sum'),  # Total runs conceded
        lbw_or_bowled=('dismissal', lambda x: ((x == 'lbw') | (x == 'bowled')).sum()),
        dot_balls=('bowlruns', lambda x: (x == 0).sum()),  # Dot balls where no runs were scored
    ).reset_index()

    # Calculate economy rate
    summary['economy'] = summary['runs'] / summary['overs'].apply(lambda x: int(x.split('.')[0]) + int(x.split('.')[1]) / 6)

    # Calculate maidens
    maiden_overs = df.groupby(['p_match', 'p_bowl', 'inns', 'over']).agg(
        over_runs=('bowlruns', 'sum')
    ).reset_index()
    maidens = maiden_overs.groupby(['p_match', 'p_bowl', 'inns'])['over_runs'].apply(lambda x: (x == 0).sum()).reset_index()
    summary = summary.merge(maidens, on=['p_match', 'p_bowl', 'inns'], how='left')
    summary.rename(columns={'over_runs': 'maidens'}, inplace=True)

    # Add venue and date
    match_info = df.groupby('p_match').agg(venue=('ground', 'first'), date=('date', 'first')).reset_index()
    summary = summary.merge(match_info, on='p_match', how='left')

    # Determine the first over bowled by each player in each innings
    first_over = df[df['legal_delivery']].groupby(['p_match', 'p_bowl', 'inns'])['over'].min().reset_index()
    summary = summary.merge(first_over, on=['p_match', 'p_bowl', 'inns'], how='left')

    # Sort by innings and the order in which they bowled their first over
    summary = summary.sort_values(by=['p_match', 'inns', 'over']).drop(columns=['over'])

    return summary

bowling_card = calculate_bowling_card(df)

In [152]:
bowling_card = bowling_card.rename(columns = {'p_match':'match_id', 'p_bowl':'bowler_id', 'bowl':'bowler', 'team_bowl':'bowl_team', 'team_bat':'bat_team',
                               'dot_balls':'dots', 'economy':'eco','maidens':'maiden'})

In [153]:
bowling_card = bowling_card.reindex(columns=[ 'date', 'match_id', 'bowl_team', 'bat_team', 'bowler_id', 'bowler',  'inns',
       'overs', 'runs', 'wickets', 'maiden', 'runs', 'eco', 'balls', 'lbw_or_bowled', 'dots', 'venue'])

In [154]:
bowling_card['venue'] = bowling_card['venue'].str.split(',').str.get(0)

In [159]:
bowling_card[bowling_card['match_id'] == 829705]

,date,match_id,bowl_team,bat_team,bowler_id,bowler,inns,overs,runs,wickets,maiden,runs,eco,balls,lbw_or_bowled,dots,venue
10,2015-04-08,829705,Kolkata Knight Riders,Mumbai Indians,376116,Umesh Yadav,1,3.0,36,0,0,36,12.000000,18,0,6,Eden Gardens
4,2015-04-08,829705,Kolkata Knight Riders,Mumbai Indians,46538,Morne Morkel,1,4.0,18,2,1,18,4.500000,24,0,15,Eden Gardens
6,2015-04-08,829705,Kolkata Knight Riders,Mumbai Indians,56143,Shakib Al Hasan,1,4.0,48,1,0,48,12.000000,26,0,7,Eden Gardens
7,2015-04-08,829705,Kolkata Knight Riders,Mumbai Indians,230558,Sunil Narine,1,4.0,28,0,0,28,7.000000,24,0,10,Eden Gardens
8,2015-04-08,829705,Kolkata Knight Riders,Mumbai Indians,276298,Andre Russell,1,3.0,21,0,0,21,7.000000,18,0,7,Eden Gardens
2,2015-04-08,829705,Kolkata Knight Riders,Mumbai Indians,32966,Piyush Chawla,1,2.0,16,0,0,16,8.000000,12,0,3,Eden Gardens
5,2015-04-08,829705,Mumbai Indians,Kolkata Knight Riders,49758,Lasith Malinga,2,4.0,27,0,0,27,6.750000,25,0,12,Eden Gardens
3,2015-04-08,829705,Mumbai Indians,Kolkata Knight Riders,35731,Vinay Kumar,2,3.3,21,0,0,21,6.000000,21,0,11,Eden Gardens
9,2015-04-08,829705,Mumbai Indians,Kolkata Knight Riders,277662,Corey Anderson,2,2.0,21,1,0,21,10.500000,12,0,5,Eden Gardens
11,2015-04-08,829705,Mumbai Indians,Kolkata Knight Riders,625383,Jasprit Bumrah,2,3.0,38,1,0,38,12.666667,19,0,7,Eden Gardens


In [156]:
bowling_card.to_csv('ganjoo_ipl_mbm_bowl', index=False)